# 街をベクトルで表すと何が変わる？
Pythonで地域Embeddingを試します。東京都内656駅・同じ1km四方・保存済みOvertureを再利用します。
Aは密度などの8特徴、Bは細分類カテゴリのTF-IDF、CはBをSVDで圧縮した64次元です。
「Embeddingの方が正しい」とは仮定しません。[実行方法](../docs/embedding.md) / [設計](../docs/embedding-design.md) / [検証](../docs/embedding-validation.md)。

In [ ]:
from pathlib import Path
import os
if Path.cwd().name == "notebooks":
    os.chdir("..")
import pandas as pd
from IPython.display import display, SVG
from geoai_open_lab.region_embedding import run_embedding
from geoai_open_lab.embedding_figures import save_embedding_figures

## 1. 固定入力を読み、次元数を比較する
入力SHA・AOI内件数・前回順位を検査してから実行します。各POIのprimaryを1カテゴリとして数え、5駅未満のカテゴリを除きます。
TFは生の件数、IDFは平滑化し、L2正規化します。同一POIは重複AOIでも同じ半分に割り当てて安定性を調べます。

In [ ]:
metrics, result, out = run_embedding(dimension=64)
display(metrics.round(4))

## 2. 吉祥寺のトップ10を並べる
距離の数値は方式間で比較できません。同名の駅グループはIDで区別します。吉祥寺と重なる候補は除きますが、候補同士の重複は残ります。

In [ ]:
tables = {mode: pd.read_csv(out / f"ranking_{mode}.csv").head(10) for mode in ("A", "B", "C64")}
display(pd.DataFrame({mode: table.station_name + " (" + table.region_id + ")" for mode, table in tables.items()}))

## 3. 同じk=3で前回の分類と比較する
64次元をそのままK-meansへ入力します。ARIは一致度であり正解率ではありません。

In [ ]:
display(pd.read_csv(out / "cluster_summary.csv"))
print("前回とのARI:", round(result["cluster_ari"], 4))

## 4. 表現・近傍・次元数・分類・潜在軸を図で確認する
潜在軸に恣意的な街の名前は付けません。係数の大きいカテゴリを併記します。

In [ ]:
figures = save_embedding_figures(out)
for path in figures[:-1]:
    display(SVG(filename=str(path)))

## 5. 変わったこと、改善しなかったこと
カテゴリを細かくすると順位は変わりますが、64次元の分割安定性はBを下回りました。
少数POIの駅、重複AOI、件数の多いカテゴリの影響があります。これは営業中店舗数や人の認識する街の正解ではありません。
前回PCAは8特徴の表示専用、今回SVDは多数のカテゴリの圧縮結果を検索・分類へ使います。両者は数学的に近い線形法ですが、入力と用途が異なります。
本実験には密度をB/Cへ直接入れていない効果と圧縮の効果が混在します。A対BとB対Cを分けて読みます。
[記事引き継ぎ](../docs/embedding-article-handoff.md)に実測値と限界を記録します。